# Preprocess FlyWire Tables

One-time FlyWire preprocessing runbook. Raw table choices come from [`paths.py`](../../../src/cex/dataset/paths.py), processing logic lives in [`raw.py`](../../../src/cex/preprocessing/raw.py), and normalized output names live in [`schema.py`](../../../src/cex/dataset/schema.py).


In [ ]:
%load_ext autoreload
%autoreload 2
import os
from pathlib import Path

import cex
from cex import get_dataset
import cex.dataset.paths as dpaths
import cex.dataset.schema as dsch
import cex.preprocessing as dprep
import cex.preprocessing.io_stat as io_stat
import cex.util.tables as rt

DATASET_NAME = "flywire"
REPO_ROOT = Path(cex.__file__).resolve().parents[2]
DATA_ROOT = REPO_ROOT / "data" / DATASET_NAME
WRITE_PREPROCESSED_Q = True

flywire = get_dataset(DATASET_NAME, DATA_ROOT)
fp_download = flywire.paths.fp_download
fp_preprocessed = flywire.paths.fp_preprocessed
fw_files = {name: os.path.join(fp_download, fn) for name, fn in dpaths.FLYWIRE_DOWNLOAD_FILES.items()}
fp_download, fp_preprocessed


## Optional Download

Set `DOWNLOAD_URL` after placing the dataset archive in Dropbox. The archive is downloaded to `download_data` and extracted there. Leave it as `None` when the raw files are already present.


In [ ]:
DOWNLOAD_URL = "https://www.dropbox.com/scl/fo/a01t6njrz0hs8b7m47nys/AHSSrFfESyg903zDTCtnOx4?rlkey=gm8k4sakl70ynxoviidw2w56h&dl=1"
DOWNLOAD_ARCHIVE_NAME = "Flywire_raw_data.zip"
DOWNLOAD_OVERWRITE_Q = False

if DOWNLOAD_URL is not None:
    dprep.download_and_extract(DOWNLOAD_URL, fp_download, DOWNLOAD_ARCHIVE_NAME, DOWNLOAD_OVERWRITE_Q)


## Inspect Downloaded Tables


In [ ]:
for name, fp in fw_files.items():
    print(f"\n{name}: {fp}")
    print(rt.table_columns(fp))
    display(rt.preview_table(fp, n=2))


## Convert `cell_data.parquet`

[`build_flywire_cell_data`](../../../src/cex/preprocessing/raw.py) merges FlyWire classification, cell type, and neurotransmitter tables, then assigns normalized `id` values while preserving source `rid`.


In [ ]:
cell_classification_table = rt.read_table(fw_files["cell_classification"])
cell_table = rt.read_table(fw_files["cell_data"])
neurotransmitter_table = rt.read_table(fw_files["neurotransmitter_data"])

cell_data = dprep.build_flywire_cell_data(cell_classification_table, cell_table, neurotransmitter_table)

if WRITE_PREPROCESSED_Q:
    rt.write_table(cell_data, os.path.join(fp_preprocessed, dsch.CELL_DATA_FILE))

cell_data.head()


## Convert `type_data.parquet`

[`build_type_data`](../../../src/cex/preprocessing/raw.py) summarizes type-level attributes by majority value and applies FlyWire type neurotransmitter ground truth from [`paths.py`](../../../src/cex/dataset/paths.py).


In [ ]:
type_data = dprep.build_type_data(cell_data, dpaths.FLYWIRE_TYPE_COLUMNS, 
                                  col_name_map=dpaths.FLYWIRE_TYPE_COLUMN_RENAME, 
                                  nt_correction=dsch.TYPE_NEUROTRANSMITTER_GT)

if WRITE_PREPROCESSED_Q:
    rt.write_table(type_data, os.path.join(fp_preprocessed, dsch.TYPE_DATA_FILE))

type_data.head()

## Convert `visual_type_data.parquet`


In [ ]:
visual_cell_data = rt.read_table(fw_files["visual_cell_data"])
visual_type_data = dprep.build_type_data(visual_cell_data, 
                                         dpaths.FLYWIRE_VISUAL_TYPE_COLUMNS)
if "side" in visual_type_data: 
    visual_type_data["side"] = dprep.normalize_side_values(visual_type_data["side"])

if WRITE_PREPROCESSED_Q:
    rt.write_table(visual_type_data, os.path.join(fp_preprocessed, dsch.VISUAL_TYPE_DATA_FILE))

visual_type_data.head()

## Convert `columns_data.npz`


In [ ]:
columns_raw = rt.read_table(fw_files["columns"])
column_data = dprep.build_column_data(
    columns_raw,
    cell_data,
    rid_col="root_id",
    kept_cols=dpaths.FLYWIRE_COLUMN_COLUMNS,
)

if WRITE_PREPROCESSED_Q:
    rt.write_table(column_data, os.path.join(fp_preprocessed, dsch.COLUMN_DATA_FILE))

column_data

## Convert `synapses.parquet`

[`normalize_synapse_table`](../../../src/cex/preprocessing/raw.py) applies the Princeton full-RID offset and maps source RIDs to normalized cell IDs.


In [ ]:
synapse_source_table = rt.read_table(fw_files["synapses"], columns=list(dpaths.FLYWIRE_SYNAPSE_COLUMNS))
synapse_table = dprep.normalize_synapse_table(synapse_source_table, cell_data, 
                    pre_rid_col="pre_root_id_720575940", 
                    post_rid_col="post_root_id_720575940",
                    x_col="ctr_x", y_col="ctr_y", z_col="ctr_z",
                    rid_add=dpaths.FLYWIRE_PRINCETON_RID_ADD)

if WRITE_PREPROCESSED_Q:
    rt.write_table(synapse_table, os.path.join(fp_preprocessed, dsch.SYNAPSE_DATA_FILE))

synapse_table.head()

## Convert `cell_to_cell_syn_count.parquet`


In [ ]:
connectivity = dprep.build_connectivity_edges(synapse_table)

if WRITE_PREPROCESSED_Q:
    rt.write_table(connectivity, os.path.join(fp_preprocessed, dsch.CELL_TO_CELL_SYN_COUNT_FILE))

connectivity.head()


## Per-Cell IO Statistics

In [ ]:
RUN_IO_STAT_Q = True
IO_STAT_NUM_WORKERS = 8

if RUN_IO_STAT_Q:
    io_stat.compute_all_io_stats(
        flywire,
        stat_type=dpaths.DEFAULT_IO_STAT_TYPE,
        min_syn_per_rid=1,
        min_frac=1e-2,
        num_workers=IO_STAT_NUM_WORKERS,
        write_Q=True,
    )

## Aggregate Type Connectivity

Aggregate the cell-level edge table once into a sparse type-by-type matrix. Entry `(i, j)` is the total number of synapses from `type_data['type'][i]` to `type_data['type'][j]`; the saved type list preserves that exact row and column order for direct lookup through `dataset.connectivity`.

In [ ]:
type_connectivity_data = dprep.build_type_connectivity_data(
    connectivity, cell_data, type_data['type'].values)
type_connectivity_fp = os.path.join(fp_preprocessed, dsch.TYPE_TO_TYPE_SYN_COUNT_FILE)
if WRITE_PREPROCESSED_Q:
    rt.write_table(type_connectivity_data, type_connectivity_fp)

type_connectivity_fp, type_connectivity_data['shape'], len(type_connectivity_data['data'])